# 1.查询保险产品及其条款文件

In [5]:
from app.infra.database import AsyncSessionFactory
from app.modules.product.service import ProductService

async with AsyncSessionFactory() as session:
    service = ProductService(session)
    products = await service.list_products()

# 拿第一个产品来测试
product = products[0]
print(product)

Product({'name': '达尔文宝贝计划12号', 'insurer': '信美人寿相互保险社', 'clause_name': '信美相互星辰守护B款少儿重疾险条款.pdf', 'description': '117种重疾+28种中症+45种轻症，少儿重疾保障灵活。', 'max_premium': Decimal('2501.83'), 'highlights': ['117种重疾', '少儿保障', '可选投保人豁免'], 'status': 'active', 'updated_at': '2026-10-23 03:32:59', 'category': 'critical_illness', 'id': 1, 'image_url': 'https://files.huizecdn.com/file3/M00/91/12/rBYA3Gi1gZ2AXBL7AAYFJxRhbPg411.png', 'min_premium': Decimal('285.30'), 'target_group': '0-17周岁', 'created_at': '2026-10-23 03:32:59'})


# 2.加载、解析条款文档

In [3]:
from mineru import MinerU
from app.core.config import settings

# 创建MinerU客户端。（精准模式，支持更多文档页数，需要TOKEN。从 https://mineru.net 获取免费代币）
client = MinerU(settings.rag.mineru_token)
result = client.extract(f"../data/raw/kb/{product.clause_name}")

In [4]:
# 三、文档优化
from langchain.chat_models import init_chat_model

MARKDOWN_OPTIMIZATION_PROMPT = """
下面这份Markdown文档是从保险条款PDF解析得来，由于PDF中的各个小节是以表格形式存在，所以解析时出现错乱。你分析内容，帮我转为格式正确的Markdown，特别是标题编号要正确。
- 标题等级要从1级标题开始，逐层增加，目录和文档名不计入标题等级。
- 输出结果中不要包含正文开始之前的部分。
- 输出结果只包含Markdown正文，不要解释或代码围栏。
- 不要修改保险条款。
""".strip()

llm = init_chat_model(
    "deepseek-v4-flash",
    api_key=settings.llm.api_key,
    extra_body={'thinking': {'type': 'disabled'}},
    max_tokens=1000  # 测试只切1000 tokens，后续可以往上调以获得完全文档
)

response = llm.invoke([
    {"role": "system", "content": MARKDOWN_OPTIMIZATION_PROMPT},
    {"role": "user", "content": result.markdown},
])
markdown = response.content

# 3.切分文档

In [5]:
# 四、文档切分
from uuid import uuid4
from langchain_core.documents import Document
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from app.rag.models import ParentChunk

# 1.准备TextSpliter
# 1.1.markdown切分器，切父块
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "h1"),
        ("##", "h2"),
        ("###", "h3"),
        ("####", "h4"),
    ],
    strip_headers=True,
)
# 1.2.递归切分器，切子块
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", "。", "；", ";", "，", ","],
    keep_separator="end",
)

# 2.先用markdown切分，得到父块小节信息
sections = markdown_splitter.split_text(markdown)
# 3.处理父子块
parent_chunks: list[ParentChunk] = []
child_chunks: list[Document] = []

# 3.1.遍历父块小节
for section in sections:
    # 3.1.生成章节路径，section.metadata就是每一级标题的字典，values刚好得到标题列表
    section_path = list(section.metadata.values())
    # 3.2.组织 parent_chunk
    parent = ParentChunk(
        id=uuid4(),
        product_id=product.id,
        content=section.page_content,
        section_path=section_path,
        clause_name=product.clause_name
    )
    parent_chunks.append(parent)

    # 3.3.重新组织section的metadata，会作为所有子块的metadata
    section.metadata = {
        "product_id": product.id,
        "parent_id": str(parent.id)
    }
    # 3.4.切分section，生成child的Document列表，并且沿用section的metadata
    children = child_splitter.split_documents([section])
    # 3.5.把父的section_path拼接到子content中，使信息完整
    section_path_header = "\n".join(section_path) + "\n"
    for c in children:
        c.page_content = section_path_header + c.page_content
    child_chunks.extend(children)

# 4.存入向量库、数据库

In [2]:
# 五、存入向量库
from langchain_ollama import OllamaEmbeddings
from langchain_milvus import Milvus, BM25BuiltInFunction
from app.core.config import settings

# 1.初始化向量模型
embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b",  # 性价比高的模型
    dimensions=1024  # 可选：减少维度以节省存储
)

# 2、初始化 VectorStore，会自动创建Collection
vector_store = Milvus(
    embedding_function=embeddings,
    collection_name="insurance_collection",
    builtin_function=BM25BuiltInFunction(analyzer_params={"type": "chinese"}),
    vector_field=["dense", "sparse"],
    connection_args={"uri": settings.rag.milvus_url},
    auto_id=True,
)

# 3.删除旧文档
vector_store.delete(expr=f"product_id=={product.id}")
# 4.添加新文档
vector_store.add_documents(child_chunks)

In [3]:
# 六、父块入数据库
from app.infra.database import AsyncSessionFactory
from app.rag.repository import ParentChunkRepository

async with AsyncSessionFactory() as session:
    async with session.begin():
        repository = ParentChunkRepository(session)
        # 先删除旧数据
        await repository.delete_by_product_id(product.id)
        # 再添加新数据
        repository.add_all(parent_chunks)

NameError: name 'product' is not defined

## 测试 检索-实现检索和召回Retriever

In [6]:
# 八、检索
from app.rag.retriever import ParentChunkRetriever

retriever = ParentChunkRetriever(vector_store)

parent_chunks = await retriever.retrieve(
    query="保险合同的等待期是多久？",
    k=5,
    product_id=product.id,
)

for parent_chunk in parent_chunks:
    print("=" * 80)
    print("product_id =", parent_chunk.product_id)
    print("clause_name =", parent_chunk.clause_name)
    print("section_path =", parent_chunk.section_path)
    print("content =", parent_chunk.content)

2026-11-05 10:32:00 [info     ] RAG Retriever 初始化完成✅️         
product_id = 1
clause_name = 信美相互星辰守护B款少儿重疾险条款.pdf
section_path = ['1. 我们保什么', '1.4 等待期']
content = 自本合同生效(或最后复效)之日起180日(含第180日)为等待期。  
被保险人在等待期内确诊非因意外伤害(见9.1)导致患有本合同所定义的重度疾病,我们不承担保险责任,但向您无息退还已交纳的本合同的保险费,本合同终止。  
被保险人在等待期内确诊非因意外伤害导致患有本合同所定义的中度疾病,我们不承担保险责任,中度疾病保险金、首次中度疾病意外保险金、中度疾病豁免保险费、首次中度疾病关爱保险金(若选择投保首次疾病关爱保险金)和首次中度疾病陪护保险金(若选择投保首次疾病陪护保险金)责任终止,本合同继续有效。  
被保险人在等待期内确诊非因意外伤害导致患有本合同所定义的轻度疾病,我们不承担保险责任,轻度疾病保险金、首次轻度疾病意外保险金、轻度疾病豁免保险费、首次轻度疾病关爱保险金(若选择投保首次疾病关爱保险金)和首次轻度疾病陪护保险金(若选择投保首次疾病陪护保险金)责任终止,本合同继续有效。  
被保险人在等待期内确诊患有本合同所定义的严重肥胖特定合并症(见9.2),我们不承担保险责任,严重肥胖手术关爱保险金责任终止,本合同继续有效。  
被保险人在等待期内确诊非因意外伤害导致患有本合同所定义的严重继发性脊柱侧弯(见9.3),我们不承担保险责任,脊柱侧弯矫正手术(见9.4)关爱保险金责任终止,本合同继续有效。  
被保险人在等待期内非因意外伤害导致身故或全残(见9.5)(若选择投保身故或全残保险金),我们不承担保险责任,但向您无息退还已交纳的本合同的保险费,本合同终止。  
被保险人因意外伤害导致保险事故的无等待期。  
本合同所定义的重度疾病、中度疾病和轻度疾病分别指载明于本合同"8.1 重度疾病"、"8.2 中度疾病"和"8.3 轻度疾病"中的疾病、疾病状态或手术。
product_id = 1
clause_name = 信美相互星辰守护B款少儿重疾险条款.pdf
section_path = ['1. 我们保什么', '1.3 保险期间']
c